## Dependencies

In [ ]:
import os
import gc
import glob
import pywt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.optimize import minimize
from sklearn.model_selection import GroupKFold
from sklearn.metrics import root_mean_squared_error
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings("ignore")

## Data ingestion and Feature engineering 

In [ ]:

def process_well_data(base_dir, typewell_dir, core_feature_cols, is_training=True):
    hz_files = glob.glob(os.path.join(base_dir, "*__horizontal_well.csv"))
    if not hz_files:
        hz_files = glob.glob(os.path.join(base_dir, "**", "*__horizontal_well.csv"), recursive=True)
        
    tw_files = glob.glob(os.path.join(typewell_dir, "*__typewell.csv"))
    if not tw_files:
        tw_files = glob.glob(os.path.join(typewell_dir, "**", "*__typewell.csv"), recursive=True)

    if len(hz_files) == 0:
        raise ValueError(f"No horizontal well files found inside path location: {base_dir}")

    typewell_dict = {}
    for f in tw_files:
        tw_df = pd.read_csv(f)
        well_name = str(os.path.basename(f).split('__')[0])
        tw_feats = [c for c in core_feature_cols if c in tw_df.columns]
        if "Z" in tw_df.columns:
            typewell_dict[well_name] = tw_df.sort_values("Z")[["Z"] + tw_feats].rename(columns={c: f"tw_{c}" for c in tw_feats})
            typewell_dict[well_name]["tw_Z"] = typewell_dict[well_name]["Z"]

    frames = []
    for f in hz_files:
        df = pd.read_csv(f)
        well_name = str(os.path.basename(f).split('__')[0])
        df['well_id'] = well_name

        if not is_training:
            df['id'] = well_name + '_' + df.index.astype(str)

        df["X_step"] = df["X"].diff().fillna(0.0)
        df["Y_step"] = df["Y"].diff().fillna(0.0)
        df["Z_step"] = df["Z"].diff().fillna(0.0)
        df["dist"] = np.sqrt(df["X_step"]**2 + df["Y_step"]**2 + df["Z_step"]**2).cumsum()

        valid_mask = df["TVT_input"].notna() if "TVT_input" in df.columns else (df["TVT"].notna() if "TVT" in df.columns else pd.Series([False]*len(df)))
        if valid_mask.any():
            last_known_idx = df[valid_mask].index[-1]
            anchor_tvt = df.loc[last_known_idx, "TVT_input" if "TVT_input" in df.columns else "TVT"]
            anchor_Z = df.loc[last_known_idx, "Z"]
        else:
            anchor_tvt = df["Z"].iloc[0]
            anchor_Z = df["Z"].iloc[0]

        df["tvt_anchor"] = anchor_tvt
        df["Z_from_anchor"] = df["Z"] - anchor_Z

        # Discrete Wavelet Transform Trend Extractions
        coeffs = pywt.wavedec(df["GR"].values, 'db4', level=4)
        cA4 = coeffs[0]
        df["GR_dwt_base_trend"] = np.interp(np.linspace(0, len(cA4) - 1, len(df)), np.arange(len(cA4)), cA4)
        for lvl in range(1, 5):
            cD = coeffs[lvl]
            df[f"GR_dwt_detail_L{5-lvl}"] = np.interp(np.linspace(0, len(cD) - 1, len(df)), np.arange(len(cD)), cD)
        
        df["GR_dwt_energy_L1"] = df["GR_dwt_detail_L1"] ** 2
        df["GR_dwt_energy_L2"] = df["GR_dwt_detail_L2"] ** 2
        df["GR_base_trend_diff"] = df["GR_dwt_base_trend"].diff().fillna(0.0)

        for c in core_feature_cols:
            df[f"{c}_diff"] = df[c].diff().fillna(0.0)

        df["GR_rolling_std_5"] = df["GR"].rolling(window=5, min_periods=1).std().fillna(0.0)
        df["GR_rolling_mean_5"] = df["GR"].rolling(window=5, min_periods=1).mean().fillna(0.0)
        df["GR_gradient"] = np.gradient(df["GR"].values)

        df["original_order_idx"] = np.arange(len(df))
        
        if well_name in typewell_dict:
            df = pd.merge_asof(df.sort_values("Z"), typewell_dict[well_name], on="Z", direction="nearest")
            df = df.sort_values("original_order_idx").drop(columns=["original_order_idx"])
        else:
            for c in core_feature_cols:
                df[f"tw_{c}"] = df[c]
            df["tw_Z"] = df["Z"]
            df = df.drop(columns=["original_order_idx"])

        # Integrated Stratigraphic Tracking Framework (Kim's Alignment Strategy)
        df["structural_drift"] = df["Z"] - df["tw_Z"]
        df["GR_local_corr_5"] = df["GR"].rolling(5, min_periods=1).corr(df["tw_GR"]).fillna(0.0)
        df["GR_local_corr_15"] = df["GR"].rolling(15, min_periods=1).corr(df["tw_GR"]).fillna(0.0)
        df["hz_slope_vs_tw_slope"] = np.gradient(df["GR"].values) - np.gradient(df["tw_GR"].values)
        
        df["GR_tw_match_delta"] = np.abs(df["GR"] - df["tw_GR"])
        df["strat_dip_velocity"] = np.gradient(df["Z"].values) - np.gradient(df["tw_Z"].values)
        df["GR_local_variance_ratio"] = df["GR_rolling_std_5"] / (df["GR"].rolling(window=25, min_periods=1).std().fillna(0.0) + 1e-5)

        df["Z_delta_cumsum"] = df["Z_step"].cumsum()
        entry_tw_Z = df["tw_Z"].iloc[0] if "tw_Z" in df.columns else df["Z"].iloc[0]
        df["Z_sequence_offset"] = (df["Z"] - entry_tw_Z) + df["Z_delta_cumsum"]
        
        df["GR_trend_long"] = df["GR"].rolling(window=50, min_periods=1).mean()
        df["GR_trend_short"] = df["GR"].rolling(window=10, min_periods=1).mean()
        df["GR_sequence_ratio"] = df["GR_trend_short"] / (df["GR_trend_long"] + 1e-5)

        frames.append(df)

    m_df = pd.concat(frames, axis=0, ignore_index=True)
    del frames
    gc.collect()

    diff_feats = [f"{c}_diff" for c in core_feature_cols]
    tw_feats = [f"tw_{c}" for c in core_feature_cols]

    rolling_feats = []
    for shift in [1, 2, 3, 5]:
        for c in core_feature_cols:
            col_name = f"{c}_lag_{shift}"
            m_df[col_name] = m_df.groupby("well_id")[c].shift(shift)
            m_df[col_name] = m_df.groupby("well_id")[col_name].bfill()
            rolling_feats.append(col_name)

    m_df["relative_Z_offset"] = m_df["Z"] - m_df["tw_Z"]
    m_df["GR_per_relative_Z"] = m_df["GR"] / (m_df["relative_Z_offset"].abs() + 1e-5)
    m_df["GR_tw_diff"] = m_df["GR"] - m_df["tw_GR"]
    
    well_means = m_df.groupby("well_id")["GR"].transform("mean")
    well_stds = m_df.groupby("well_id")["GR"].transform("std")
    m_df["GR_well_normalized"] = (m_df["GR"] - well_means) / (well_stds + 1e-5)

    new_geology_feats = [
        "GR_rolling_std_5", "GR_rolling_mean_5", "GR_gradient", 
        "relative_Z_offset", "GR_well_normalized", "Z_delta_cumsum", 
        "Z_sequence_offset", "GR_trend_long", "GR_trend_short", "GR_sequence_ratio",
        "GR_dwt_base_trend", "GR_dwt_detail_L1", "GR_dwt_detail_L2", 
        "GR_dwt_detail_L3", "GR_dwt_detail_L4", "GR_dwt_energy_L1", 
        "GR_dwt_energy_L2", "GR_base_trend_diff", "structural_drift",
        "GR_local_corr_5", "GR_local_corr_15", "hz_slope_vs_tw_slope",
        "Z_from_anchor", "GR_tw_match_delta", "strat_dip_velocity", "GR_local_variance_ratio"
    ]
    interaction_feats = ["GR_per_relative_Z", "GR_tw_diff"]

    full_feats = core_feature_cols + diff_feats + tw_feats + rolling_feats + interaction_feats + new_geology_feats + ["dist", "tvt_anchor"]
    m_df[full_feats] = m_df.groupby("well_id")[full_feats].ffill().bfill()

    if is_training:
        m_df["TVT"] = m_df.groupby("well_id")["TVT"].ffill().bfill().fillna(0.0)

    return m_df, full_feats

def inject_global_spatial_prior(train_df, test_df, k_neighbors=8):
    train_coords = train_df[['X', 'Y', 'Z']].values
    spatial_tree = cKDTree(train_coords)
    
    train_dists, train_indices = spatial_tree.query(train_coords, k=k_neighbors + 1)
    train_dists = train_dists[:, 1:]
    train_indices = train_indices[:, 1:]
    
    train_weights = 1.0 / (train_dists + 1e-5)
    train_norm_weights = train_weights / np.sum(train_weights, axis=1, keepdims=True)
    
    train_neighbor_residuals = train_df['TVT_residual'].values[train_indices]
    train_df['spatial_consensus_prior'] = np.sum(train_neighbor_residuals * train_norm_weights, axis=1)
    
    test_coords = test_df[['X', 'Y', 'Z']].values
    test_dists, test_indices = spatial_tree.query(test_coords, k=k_neighbors)
    
    test_weights = 1.0 / (test_dists + 1e-5)
    test_norm_weights = test_weights / np.sum(test_weights, axis=1, keepdims=True)
    
    test_neighbor_residuals = train_df['TVT_residual'].values[test_indices]
    test_df['spatial_consensus_prior'] = np.sum(test_neighbor_residuals * test_norm_weights, axis=1)
    
    return train_df, test_df

FEATURE_COLS = ["X", "Y", "Z", "GR"]
TRAIN_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/train"  
TEST_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/test"

train_df, ENGIN_FEATS = process_well_data(TRAIN_DIR, TRAIN_DIR, FEATURE_COLS, is_training=True)
test_df, _ = process_well_data(TEST_DIR, TEST_DIR, FEATURE_COLS, is_training=False)

train_df["TVT_residual"] = train_df["TVT"] - train_df["tvt_anchor"]
train_df, test_df = inject_global_spatial_prior(train_df, test_df)

DROP_FEATS = ['X', 'Y', 'Z', 'tw_X', 'tw_Y', 'tw_Z', 'X_lag_1', 'X_lag_2', 'X_lag_3', 'X_lag_5', 
              'Y_lag_1', 'Y_lag_2', 'Y_lag_3', 'Y_lag_5', 
              'Z_lag_1', 'Z_lag_2', 'Z_lag_3', 'Z_lag_5', 'tvt_anchor', 'Z_diff']

GEO_FEATS = [c for c in ENGIN_FEATS if c not in DROP_FEATS]
if 'spatial_consensus_prior' not in GEO_FEATS:
    GEO_FEATS.append('spatial_consensus_prior')

X_base = train_df[GEO_FEATS].values
y_res = train_df["TVT_residual"].values
y_true = train_df["TVT"].values
groups = train_df["well_id"].values
X_test = test_df[GEO_FEATS].values

print("--- Integrated Alignment Data Processing Complete ---")
print(f"Train Matrix Shape: {X_base.shape} | Test Matrix Shape: {X_test.shape}")
print(f"Total Aligned Stratigraphic Features: {len(GEO_FEATS)}")

In [ ]:
print(train_df[['TVT', 'tvt_anchor', 'TVT_residual']].describe())

## Training (using an ensemble of Catboost, lightgbm and xgboost)

In [ ]:
os.makedirs("/kaggle/working/best_models", exist_ok=True)

PARAMS = {
    'lgb': {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': 0.015,
        'num_leaves': 127,
        'max_depth': 10,
        'min_data_in_leaf': 20,
        'feature_fraction': 0.8,
        'verbosity': -1,
        'random_state': 42, 
        'device': 'gpu' # Switches LightGBM to GPU acceleration
    },
    'xgb': {
        'n_estimators': 2000,
        'learning_rate': 0.02,
        'max_depth': 8,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'early_stopping_rounds': 100, 
        'tree_method': 'hist', # Modern fast histogram calculation method
        'device': 'cuda' # Utilizes the GPU core layers
    },
    'cat': {
        'iterations': 2000,
        'learning_rate': 0.03,
        'depth': 8,
        'loss_function': 'RMSE',
        'random_seed': 42,
        'verbose': 0, 
        'early_stopping_rounds': 100, 
        'task_type': 'GPU' # Switches CatBoost engine to GPU mode
    }
}

gkf = GroupKFold(n_splits=5)
oofs_res = {m: np.zeros(len(train_df)) for m in ['lgb', 'xgb', 'cat']}
fold_metrics = {m: [] for m in ['lgb', 'xgb', 'cat']}
trained_models = {m: [] for m in ['lgb', 'xgb', 'cat']}

lgb_test_res_preds = np.zeros(len(test_df))
xgb_test_res_preds = np.zeros(len(test_df))
cat_test_res_preds = np.zeros(len(test_df))

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_base, y_res, groups=groups)):
    print(f"--- Consistently Optimizing Fold {fold + 1} ---")
    X_tr, X_v = X_base[tr_idx], X_base[val_idx]
    y_tr, y_v = y_res[tr_idx], y_res[val_idx]

    # LightGBM
    model_lgb = lgb.train(PARAMS['lgb'], lgb.Dataset(X_tr, y_tr, feature_name=GEO_FEATS), 2500, valid_sets=[lgb.Dataset(X_v, y_v, feature_name=GEO_FEATS)], callbacks=[lgb.early_stopping(100, verbose=False)])
    lgb_preds = model_lgb.predict(X_v)
    oofs_res['lgb'][val_idx] = lgb_preds
    lgb_test_res_preds += model_lgb.predict(X_test) / 5.0
    fold_metrics['lgb'].append(root_mean_squared_error(y_v, lgb_preds))
    trained_models['lgb'].append(model_lgb)
    
    # XGBoost
    model_xgb = xgb.XGBRegressor(**PARAMS['xgb']).fit(X_tr, y_tr, eval_set=[(X_v, y_v)], verbose=False)
    xgb_preds = model_xgb.predict(X_v)
    oofs_res['xgb'][val_idx] = xgb_preds
    xgb_test_res_preds += model_xgb.predict(X_test) / 5.0
    fold_metrics['xgb'].append(root_mean_squared_error(y_v, xgb_preds))
    trained_models['xgb'].append(model_xgb)
    
    # CatBoost
    model_cat = CatBoostRegressor(**PARAMS['cat']).fit(X_tr, y_tr, eval_set=(X_v, y_v), verbose=False)
    cat_preds = model_cat.predict(X_v)
    oofs_res['cat'][val_idx] = cat_preds
    cat_test_res_preds += model_cat.predict(X_test) / 5.0
    fold_metrics['cat'].append(root_mean_squared_error(y_v, cat_preds))
    trained_models['cat'].append(model_cat)

best_model_name = min(['lgb', 'xgb', 'cat'], key=lambda m: np.mean(fold_metrics[m]))
best_fold_idx = np.argmin(fold_metrics[best_model_name])
winning_model = trained_models[best_model_name][best_fold_idx]

with open("/kaggle/working/best_models/best_choice.txt", "w") as f:
    f.write(best_model_name)

winning_model.save_model("/kaggle/working/best_models/best.model")

print("\n--- Real TVT Object Target OOF Scores ---")
anchors = train_df["tvt_anchor"].values

for model_name, oof_res_preds in oofs_res.items():
    real_tvt_pred = oof_res_preds + anchors
    score = root_mean_squared_error(y_true, real_tvt_pred)
    print(f"{model_name.upper()} Real TVT RMSE: {score:.4f}")

# Target Optimization Engine
def absolute_tvt_rmse_func(weights):
    w_lgb, w_xgb, w_cat = weights
    blended_residual = (w_lgb * oofs_res['lgb']) + (w_xgb * oofs_res['xgb']) + (w_cat * oofs_res['cat'])
    blended_absolute_tvt = blended_residual + anchors
    return root_mean_squared_error(y_true, blended_absolute_tvt)

starting_weights = [1/3, 1/3, 1/3]
bounds = [(0, 1), (0, 1), (0, 1)]
constraints = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})

res = minimize(absolute_tvt_rmse_func, starting_weights, method='SLSQP', bounds=bounds, constraints=constraints)
best_w = res.x

print(f"\n--- Best Ensemble Optimization Results ---")
print(f"Optimized Weight Matrix -> LGB: {best_w[0]:.4f} | XGB: {best_w[1]:.4f} | CAT: {best_w[2]:.4f}")
print(f"Optimized Blended TVT Target RMSE: {res.fun:.4f}")

## Feature importance


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

with open("/kaggle/working/best_models/best_choice.txt", "r") as f:
    best_model_name = f.read().strip()

print(f"Plotting importance for the winning ensemble component: {best_model_name.upper()}")

if best_model_name == 'lgb':
    model = lgb.Booster(model_file="/kaggle/working/best_models/best.model")
    imp_scores = model.feature_importance(importance_type='gain')
    
elif best_model_name == 'xgb':
    model = xgb.XGBRegressor()
    model.load_model("/kaggle/working/best_models/best.model")
    booster = model.get_booster()
    importance_dict = booster.get_score(importance_type='gain')
    imp_scores = [importance_dict.get(f'f{i}', 0.0) for i in range(len(GEO_FEATS))]
    
else:
    model = CatBoostRegressor()
    model.load_model("/kaggle/working/best_models/best.model")
    imp_scores = model.get_feature_importance()

imp_scores = np.array(imp_scores)
imp_scores_norm = imp_scores / (imp_scores.sum() + 1e-5)
importance_df = pd.DataFrame({
    'Feature': GEO_FEATS,
    'Importance_Pct': imp_scores_norm * 100
}).sort_values(by='Importance_Pct', ascending=False)

print(f"\n=== TOP 15 FEATURES ({best_model_name.upper()}) ===")
print(importance_df.head(15).to_string(index=False, formatters={'Importance_Pct': '{:,.2f}%'.format}))

plt.figure(figsize=(12, 10))
sns.barplot(
    data=importance_df.head(25),
    x='Importance_Pct', 
    y='Feature', 
    hue='Feature', 
    palette='viridis', 
    legend=False
)
plt.title(f'Top 25 Feature Importances: {best_model_name.upper()}\n(Integrated Stratigraphic Alignment Model)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Normalized Gain Importance (%)', fontsize=12)
plt.ylabel('Engineered Stratigraphic Feature', fontsize=12)
plt.grid(axis='x', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

The previous cells show how i made the program. this cell is for generating submissions.csv using the ensemble model

In [ ]:
final_test_residual = (best_w[0] * lgb_test_res_preds) + (best_w[1] * xgb_test_res_preds) + (best_w[2] * cat_test_res_preds)
final_test_tvt_preds = final_test_residual + test_df["tvt_anchor"].values

test_df['local_row_idx'] = test_df.groupby('well_id').cumcount()
test_df['id'] = test_df['well_id'].astype(str) + '_' + test_df['local_row_idx'].astype(str)

prediction_map = dict(zip(test_df['id'].values, final_test_tvt_preds))

sample_sub_path = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/sample_submission.csv"
sample_sub = pd.read_csv(sample_sub_path)

sample_sub['tvt'] = sample_sub['id'].map(prediction_map)

median_backup = np.median(final_test_tvt_preds) if len(final_test_tvt_preds) > 0 else 0.0
sample_sub['tvt'] = sample_sub['tvt'].fillna(median_backup)

sample_sub['id'] = sample_sub['id'].astype(str)
sample_sub['tvt'] = sample_sub['tvt'].astype(float)

sample_sub.to_csv("submission.csv", index=False)

print("--- Strict Target Format Verification ---")
print(f"✓ Submission Row Dimension: {sample_sub.shape}")
print(f"✓ Column Layout: {list(sample_sub.columns)}")
print(f"✓ Total Blank NaN Items: {sample_sub['tvt'].isna().sum()}")
print("\nPreview:")
print(sample_sub.head())